# Episode 4 — Wavetable Position: Morphing Between Waveforms

⏱ About 25 minutes · Difficulty ★★☆

**Musical element this episode controls: how timbre changes over time.**

**The equivalent Max patch:** a `buffer~` holding many single cycles, with `wave~` reading two neighbours and crossfading. In Serum this is the **WT POS** knob.

**What you will hear:**
1. `pos = 0`: dark (sine); `pos = 1`: bright (close to a saw)
2. `pos` sliding from 0 to 1 over 2 seconds: the sound opens from "ooh" to "aah"
3. `pos` pushed back and forth by an LFO: "wah-wah"
4. a strangely shaped wavetable of your own

**By the end you can:**
- read a 2-D array (a stack of tables)
- explain how wavetable position crossfades between two tables
- make the position change over time
- build your own set of frames

## Toolbox

Everything built in earlier episodes, copied as-is. Just run it; no need to re-read (the tricky lines carry `#` comments).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip holds anything beyond ±1 so nothing blows up
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # only plot this many samples from the start
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

def make_table(harmonic_amps, N=2048):
    p = np.arange(N) / N                       # phase of every table position, 0 to 1
    table = np.zeros(N)                        # start from an empty table of zeros
    for k, amp in enumerate(harmonic_amps, start=1):
        # harmonic k: k cycles across the table, at level amp
        table += amp * np.sin(2 * np.pi * k * p)
    return table / np.max(np.abs(table))       # normalize: peak becomes 1

def osc_table(table, phase):
    N = len(table)
    pos = phase * N
    i0 = np.floor(pos).astype(int) % N
    i1 = (i0 + 1) % N
    frac = pos - np.floor(pos)
    return (1 - frac) * table[i0] + frac * table[i1]

freq = 110
dur = 2.0
t = np.arange(int(sr * dur)) / sr
phase = (freq * t) % 1

## 1. Start with two tables: a crossfade

One dark table (a sine) and one bright table (32 harmonics).

Blend them with a number `mix` between 0 and 1:

`(1 - mix) * dark + mix * bright`

- `mix = 0` → all dark
- `mix = 1` → all bright
- `mix = 0.5` → half of each

You have seen this formula before: Episode 3's interpolation is the same thing (`(1 - frac) * left + frac * right`). In Max it is `xfade~` or `mix~`.

**You should hear:** three tones at the same pitch, each brighter than the last.

In [ ]:
dark = make_table([1])
bright = make_table([1 / k for k in range(1, 33)])

for mix in [0.0, 0.5, 1.0]:
    blended = (1 - mix) * dark + mix * bright      # blend the two tables into a new one, then read that
    scope(osc_table(blended, phase), f"mix = {mix}")
    display(play(osc_table(blended, phase)))

Between two tables there is only one kind of "in between". For richer movement you need **many tables in a row**, crossfading between neighbours.

## 2. 2-D arrays

One table is one row of numbers (1-D). Many tables stacked form a grid with **rows and columns** (2-D).

Look at a small one first: 2 rows, 3 columns.

- `m.shape` → (rows, columns)
- `m[0]` → row 0 (the whole row)
- `m[1, 2]` → item 2 of row 1

In [ ]:
m = np.array([[1, 2, 3],
              [10, 20, 30]])

print("shape   =", m.shape)
print("m[0]    =", m[0])
print("m[1]    =", m[1])
print("m[1, 2] =", m[1, 2])

## 3. A stack of tables: frames

Make 16 tables, each called a **frame**:

- frame 0: 1 harmonic (a sine)
- frame 15: 32 harmonics (close to a saw)
- in between: the harmonic count rises evenly

`frames` has shape (16, 2048): 16 rows, each a 2048-point table. `frames[3]` is table number 3.

In [ ]:
def make_frames(F=16, max_harmonics=32, N=2048):
    frames = []                                    # an empty list to collect each table
    for f in range(F):                             # f = 0, 1, 2 ... 15
        # how many harmonics frame f gets: rising evenly from 1 to max_harmonics
        n_h = 1 + round(f * (max_harmonics - 1) / (F - 1))
        frames.append(make_table([1 / k for k in range(1, n_h + 1)], N))
    return np.array(frames)                        # stack into a 2-D array: F rows × N columns

frames = make_frames()
print("shape:", frames.shape)

All 16 frames, spread out from bottom to top (each shifted up a little):

In [ ]:
plt.figure(figsize=(9, 6))
for f in range(len(frames)):
    plt.plot(frames[f] + 1.5 * f, color="C0")      # + 1.5 * f only spreads them out in the plot; the tables are unchanged
plt.yticks(1.5 * np.arange(len(frames)), [f"frame {f}" for f in range(len(frames))])
plt.xlabel("table index")
plt.show()

Listen to a few on their own. **You should hear:** frame 0 darkest, frame 15 brightest, the ones between getting gradually brighter.

In [ ]:
for f in [0, 3, 8, 15]:
    print("frame", f)
    display(play(osc_table(frames[f], phase)))

## 4. Position

`pos` is a number from 0 to 1: 0 = the first frame, 1 = the last.

What about values in between? First convert to a "frame number" that may have decimals:

`fpos = pos * (F - 1)`

With 16 frames, `F - 1 = 15`:

| `pos` | `fpos` | meaning |
|---|---|---|
| 0 | 0 | exactly frame 0 |
| 1 | 15 | exactly frame 15 |
| 0.5 | 7.5 | half frame 7, half frame 8 |
| 0.3 | 4.5 | half frame 4, half frame 5 |
| 0.1 | 1.5 | half frame 1, half frame 2 |

Then exactly as when reading a table in Episode 3: round down for "the frame below", add 1 for "the frame above", and the fractional part sets the blend.

In [ ]:
F = len(frames)

for pos in [0.0, 0.1, 0.5, 0.98, 1.0]:
    fpos = pos * (F - 1)
    f0 = int(np.floor(fpos))               # the frame below
    f1 = min(f0 + 1, F - 1)                # the frame above; min keeps it from passing the last one
    ffrac = fpos - f0                      # how much of the upper frame
    print(f"pos = {pos:<5} fpos = {fpos:<6.2f} frame {f0} x {1 - ffrac:.2f}  +  frame {f1} x {ffrac:.2f}")

## 5. The complete wavetable oscillator

There are now **two layers of interpolation**:

1. **inside** one table, between two neighbouring samples (Episode 3)
2. **between two tables** (this episode)

`frames[f0, i0]` means: position `i0` of table number `f0`.

The function has three parts, each with a comment in front.

In [ ]:
def osc_wavetable(frames, phase, pos):
    F, N = frames.shape

    # --- which two frames are we between ---
    fpos = np.clip(pos, 0, 1) * (F - 1)        # turn 0..1 into a frame number 0..F-1, decimals allowed
    f0 = np.floor(fpos).astype(int)            # the frame below
    f1 = np.minimum(f0 + 1, F - 1)             # the frame above (never past the last one)
    ffrac = fpos - f0                          # how far above the lower frame, 0..1

    # --- which two positions inside a table (same as Episode 3) ---
    p = phase * N
    i0 = np.floor(p).astype(int) % N           # left position
    i1 = (i0 + 1) % N                          # right position, wrapping to 0 at the end
    frac = p - np.floor(p)                     # the fractional part

    # --- interpolate inside each frame first, then between the two frames ---
    a = (1 - frac) * frames[f0, i0] + frac * frames[f0, i1]     # the value read from the lower frame
    b = (1 - frac) * frames[f1, i0] + frac * frames[f1, i1]     # the value read from the upper frame
    return (1 - ffrac) * a + ffrac * b                          # crossfade the two

### Listen: a fixed position

**You should hear:** three tones at the same pitch, each brighter than the last.

In [ ]:
for pos in [0.0, 0.3, 1.0]:
    print("pos =", pos)
    display(play(osc_wavetable(frames, phase, pos)))

## 6. A moving position

This is the most important step of the episode.

Above, `pos` was **one number**: the whole 2 seconds stayed in one place.

`pos` can also be **an array as long as the sound**: each sample gets its own position. The function needs no change at all — every operation inside it (multiply, round, look up) already works on whole arrays.

In Max this is the difference between patching a number into the WT POS inlet and patching a signal into it.

### 6.1 `line~`: a straight sweep

`np.linspace(0, 1, len(t))`: walk evenly from 0 to 1 in `len(t)` steps. That is `line~`.

**You should hear:** one note opening smoothly from "ooh" (dark) to "aah" (bright) over 2 seconds.

In [ ]:
pos = np.linspace(0, 1, len(t))        # one position per sample: 0, ..., 1

print("pos is an array of length", len(pos), "  first:", pos[0], "  last:", pos[-1])
play(osc_wavetable(frames, phase, pos))

Look at it as a spectrogram (time across, frequency up, brighter = louder; Episode 10 covers this properly).

**You should see:** only the bottom line (the fundamental) at first, then the harmonics above it appearing one by one.

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.specgram(osc_wavetable(frames, phase, pos) + 1e-9, NFFT=2048, Fs=sr, noverlap=1536, cmap="magma", vmin=-120)
plt.ylim(0, 4000)
plt.xlabel("time (s)")
plt.ylabel("frequency (Hz)")
plt.show()

### 6.2 An LFO: back and forth

A sine that goes back and forth twice per second. `np.sin` ranges from -1 to 1 but position needs 0 to 1, hence `0.5 + 0.5 * sin`.

**You should hear:** the timbre opening and closing, "wah-wah", twice per second.

In [ ]:
lfo = np.sin(2 * np.pi * 2 * t)        # a 2 Hz sine, -1..1
pos = 0.5 + 0.5 * lfo                  # shifted to 0..1

plt.figure(figsize=(9, 2))
plt.plot(t, pos)
plt.xlabel("time (s)")
plt.ylabel("pos")
plt.show()

play(osc_wavetable(frames, phase, pos))

### 6.3 A quick fall

`np.exp(-6 * t)`: starts at 1, drops fast, then slower and slower (exponential decay).

**You should hear:** bright at the start, dark within half a second — a "pew" feel.

In [ ]:
pos = np.exp(-6 * t)

plt.figure(figsize=(9, 2))
plt.plot(t, pos)
plt.xlabel("time (s)")
plt.ylabel("pos")
plt.show()

play(osc_wavetable(frames, phase, pos))

## 7. Building your own frames

`frames` does not have to run "dark to bright". Any set of 2048-point tables stacked up will do.

Below are 8 tables whose 12 harmonic levels are **random**.

- `rng = np.random.default_rng(3)`: a random number generator. `3` is the seed — the same seed gives the same random numbers every time; another number gives another set of tables.
- `rng.uniform(0, 1, 12)`: 12 random numbers between 0 and 1
- `/ np.arange(1, 13)`: divide the k-th by k so upper harmonics are weaker overall and not too harsh

**You should hear:** as `pos` sweeps, the timbre does not simply brighten; it keeps reshaping, a bit like changing vowels.

In [ ]:
rng = np.random.default_rng(3)

weird = []
for f in range(8):
    amps = rng.uniform(0, 1, 12) / np.arange(1, 13)     # 12 random levels, weaker for higher harmonics
    weird.append(make_table(amps))
weird = np.array(weird)

fig, axes = plt.subplots(1, 8, figsize=(13, 1.8), sharey=True)
for f, ax in enumerate(axes):
    ax.plot(weird[f])
    ax.set_title(f"frame {f}", fontsize=8)
    ax.set_xticks([])
plt.show()

play(osc_wavetable(weird, phase, np.linspace(0, 1, len(t))))

## Common mistakes

| Symptom | Cause |
|---|---|
| `IndexError` saying index 16 is out of range | `pos` went above 1. The `np.clip(pos, 0, 1)` inside the function guards against this |
| `shape mismatch` | `pos` is an array whose length differs from `phase` |
| The position moves but nothing audible changes | Neighbouring frames are too alike, or `pos` covers too small a range |
| Clicks during a sweep | Neighbouring tables differ too much, or there are too few frames |

## Exercises

**1.** Let `pos` run from 1 to 0 (reversed). **You should hear:** bright to dark.

<details><summary>Answer</summary>

```python
play(osc_wavetable(frames, phase, np.linspace(1, 0, len(t))))
```
</details>

**2.** Change the LFO of 6.2 to 8 times per second and keep the position between 0.2 and 0.6. Hint: the centre is 0.4, the depth is 0.2.

<details><summary>Answer</summary>

```python
pos = 0.4 + 0.2 * np.sin(2 * np.pi * 8 * t)
play(osc_wavetable(frames, phase, pos))
```
</details>

**3.** Build a "sine to square" set of frames: 16 of them, odd harmonics only, with a rising count. Hint: the odd-harmonic list is `[1, 0, 1/3, 0, 1/5, ...]`.

<details><summary>Answer</summary>

```python
sq = []
for f in range(16):
    amps = []
    for k in range(1, 2 * f + 2):
        amps.append(1 / k if k % 2 == 1 else 0)     # 1/k for odd k, 0 for even k
    sq.append(make_table(amps))
sq = np.array(sq)
play(osc_wavetable(sq, phase, np.linspace(0, 1, len(t))))
```
</details>

## Recap

| Max / Serum | Python |
|---|---|
| `xfade~` / `mix~` | `(1 - mix) * a + mix * b` |
| a `buffer~` with many single cycles | `frames` (a 2-D array) |
| the WT POS knob | `pos` (one number) |
| a signal into WT POS | `pos` (an array) |
| `line~` | `np.linspace(start, end, n)` |

**The most important point:** in Python there is no difference between a "parameter" and a "signal" — pass one number and it is static, pass an array and it moves.

**Next episode:** `adsr~` — giving the sound a beginning and an end.